# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [7]:
%pip install -U langchain-core

  Using cached langchain_core-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.11.1-py3-none-any.whl.metadata (22 kB)
  Using cached uuid_utils-0.17.0-cp312-cp312-win_amd64.whl.metadata (6.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached orjson-3.12.0-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached xxhash-4.0.1-cp312-cp312-win_amd64.whl.metadata (18 kB)
  Using cached zstandard-0.25.0-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
Using cached langchain_core-1.6.0-py3-none-any.whl (570 kB)
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
Using cached langsmith-0.11.1-py3-none-any.whl (744 kB)
Using cached uuid_utils-0.17.0-cp312-cp312-win_amd64.whl (172 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached orjson-3.12.0-cp312-cp312-win_amd64.whl (122 kB)
Using 


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### RunnableLambda
일반 Python함수를 lcel체인에서 사용할 수 있는 Runnable 형태로 wrapping 처리해주는 클래스

In [1]:
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕 만나서 반갑다~')

11

In [2]:
runnable.batch([
    '안녕 만나서 반갑다~',
    '너도? 나도',
    '?!',
    '🤣'
])

[11, 6, 2, 1]

In [3]:
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

celsius_temps = [0, 25, 100, -10, 37]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temps)

[32.0, 77.0, 212.0, 14.0, 98.6]

In [ ]:
import time

def generator(x):
    for y in x: # 입력을 문자 단위로 순회
        yield y # 한 글자씩 반환

runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세요~😊'*3):
    print(chunk, end = '', flush = True) # chunk를 줄바꿈없이 즉시 출력
    time.sleep(0.1) # 글자 출력마다 딜레이 0.1초

안녕하세요~😊안녕하세요~😊안녕하세요~😊

In [6]:
# 사용 예시
def gen(x):
    for y in x:
        yield y

gen10 = gen(range(10))

for n in gen10:
    print(n)

0
1
2
3
4
5
6
7
8
9


In [7]:
next(gen10) # 제네레이터 다음 값 1개 반환 (다 꺼내고나면 StopIteration 발생)

StopIteration: 

### RunnableSequence
Runnable 객체를 순차 연결해주는 Runnable 객체

In [ ]:
from langchain_core.runnables import RunnableSequence # Runnable들을 순서대로 연결하는 시퀀스

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableSequence(runnable1, runnable2)
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [10]:
chain = runnable1 | runnable2
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

### RunnableParellel
여러 Runnable 객체를 인자로 받아, 병렬철 후 각각의 응답을 하나의 dict로 반환

In [11]:
from langchain_core.runnables import RunnableParallel # 여러 Runnable들을 같은 입력으로 병렬로 실행

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableParallel(r1=runnable1, r2=runnable2) # r1, r2를 병렬 실행해 dict로 변환
chain.invoke(3)

{'r1': {'foo': 3}, 'r2': [3, 3, 3]}

### 사용자가 준 주제를 이용해 삼행시, 농담, 시를 각각 생성해서 하나의 응답으로 반환

In [14]:
%pip install -U langchain langchain-openai

  Using cached langchain_openai-1.6.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.3-py3-none-any.whl.metadata (5.1 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
  Using cached tiktoken-0.14.0-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiter-0.16.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached httpcore2-2.12.0-py3-none-any.whl.metadata (25 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
Using cached langgraph-1.2.11-py3-none-any.whl (248 kB)
Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl (56 kB)
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached langgraph_sdk-0.4.3-


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_core.prompts import PromptTemplate  # prompt chain 구성
from langchain.chat_models import init_chat_model  # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser  # 답변 문자열 변환
from langchain_core.runnables import RunnableParallel, RunnableLambda  # Runnable 실행

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 박명수급 n행시의 고수입니다. 다음 주제로 n행시를 지어주세요. 주제: {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser


joke_prompt = PromptTemplate.from_template(
    '당신은 이수근급 농담의 고수입니다. 다음 주제로 농담을 지어주세요. 주제: {topic}'
)
joke_chain = joke_prompt | llm | output_parser


poem_prompt = PromptTemplate.from_template(
    '당신은 리그오브레전드 커뮤니티 따거행님급 댓글러입니다. 다음 주제로 눈물이 나오는 감성적인 시를 지어주세요. 주제: {topic}'
)
poem_chain = poem_prompt | llm | output_parser


chain = RunnableParallel(
    acrostic_poem=n_poem_chain,
    joke=joke_chain,
    poem=poem_chain
)


def combine_result(input_dict: dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem = input_dict['poem']

    return f"""
n행시:
{acrostic_poem}

농담:
{joke}

현대시:
{poem}
    """


chain = chain | RunnableLambda(combine_result)

print(chain.invoke({'topic': '런닝맨'}))


n행시:
**런닝맨 n행시**

**런**닝맨 나온다길래 운동화 신고 갔더니  
**닝**닝하게 이름표만 뜯기고  
**맨** 먼저 집에 왔다… 이게 예능이냐, 조기퇴근이지!

농담:
런닝맨 멤버들이 왜 매주 뛰는 줄 아세요?

출연료가 **시급**이 아니라 **시속**으로 계산돼서요.  
근데 유재석은 안 뛰어도 됩니다.  
**입담이 이미 결승선에 먼저 도착하거든요.** 😄

현대시:
### **〈달리고 또 달리던 우리〉**

해 질 무렵 텅 빈 운동장에  
누군가의 웃음소리가 남아 있었다  

이름표 하나 등에 붙이고  
서로를 쫓아 달리던 사람들,  

오늘은 네가 술래고  
내일은 내가 술래라며  
잡히고도 웃을 수 있었던 그 시절은  
참 이상하게 따뜻했다  

미션 하나에 목숨 걸고  
배신 하나에 세상이 무너진 듯했지만,  
결국 마지막엔 같은 차에 올라  
서로의 어깨를 빌려 잠들던 우리  

시간은 말없이 흘러  
누군가는 떠나고  
누군가는 더 이상 예전처럼 달릴 수 없게 되었지만  

화면 속 그들은 여전히 달린다  

비 오는 날에도,  
눈물 나는 날에도,  
웃을 힘조차 남지 않은 날에도  

“다음 주에 또 만나요”  
그 한마디를 남기고  
우리의 일요일을 지켜준다  

어쩌면 우리가 그토록 사랑한 건  
게임도, 벌칙도, 이름표도 아니었을 것이다  

아무 이유 없이 웃던 사람들,  
끝까지 함께 달려주던 마음,  
그리고 언젠가 사라질 걸 알면서도  
매주 기다렸던 그 시간  

런닝맨,  

당신들은 달렸고  
우리는 그 뒤를 따라 웃었다  

시간이 아무리 우리를 멀리 데려가도  
그때의 일요일만큼은  
영원히 출발선에 서 있다.
    


### RunnablePassThrough
- 사용자의 입력값을 그대로 전달해주는 Runnable

In [8]:
from langchain_core.runnables import RunnablePassthrough

n_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 고수입니다. 다음 주제로 맛깔나는 n행시를 지어주세요. 주제 : {topic}'
)
chain = {'topic': RunnablePassthrough()} | n_poem_prompt | llm | output_parser

print(chain.invoke('학원수업'))

**학원수업**

학: 학교 끝나면 또 다른 전쟁터,  
원: 원하는 건 성적 향상 하나인데  
수: 수많은 문제와 씨름하다 보니  
업: 업그레이드된 건 실력인지, 다크서클인지!


In [ ]:
prompt = PromptTemplate.from_template("""
당신은 n행시의 고수입니다. 다음 주제로 맛깔나는 n행시를 지어주세요.

주제 : {topic}

출력형식 :
===== <주제> <n행시> =====
<n행시 작성>
""")
chain = ({'topic': RunnablePassthrough()}
        | RunnablePassthrough.assign(        # 기존 topic만 있던 dict -> 새 key를 추가
            n = lambda x: len(x['topic']),   # n = topic 길이
            k = lambda x : 100               # k = 100 (미사용)
        )
        
        | prompt  # 확장된 dict(topic, n, k를)를 프롬프트에 주입하여 완성
        | llm
        | output_parser
)
print(chain.invoke('학원수업'))

===== 학원수업 4행시 =====  
**학**습하러 들어왔는데 졸음이 먼저 출석하고  
**원**장님 눈빛에 정신이 번쩍, 필기는 번개처럼  
**수**업 끝나자마자 머릿속 지식은 복습 모드  
**업**그레이드된 실력으로 시험지를 당당히 정복한다!
